In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

In [ ]:
# 1. Pre-process
df = pd.read_csv("uber.csv")
df = df.drop(columns=["Unnamed: 0", "key"])
df = df.dropna()

df["pickup_datetime"] = pd.to_datetime(df["pickup_datetime"])
df["hour"] = df["pickup_datetime"].dt.hour
df = df.drop(columns=["pickup_datetime"])

# approx distance in km (1 degree ~ 111 km)
df["distance"] = np.sqrt((df["dropoff_longitude"] - df["pickup_longitude"])**2 +
                         (df["dropoff_latitude"] - df["pickup_latitude"])**2) * 111
df.head()

In [ ]:
# 2. Identify outliers
df.boxplot(column=["fare_amount", "distance"])
plt.show()

# remove outliers using IQR
for col in ["fare_amount", "distance"]:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    df = df[(df[col] >= q1 - 1.5 * iqr) & (df[col] <= q3 + 1.5 * iqr)]
df.shape

In [ ]:
# 3. Correlation
df.corr()["fare_amount"]

In [ ]:
# 4. Train-test split
X = df.drop(columns=["fare_amount"])
y = df["fare_amount"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# 5. Linear Regression & Random Forest, evaluate with R2 and RMSE
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=50, n_jobs=-1, random_state=42),
}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    print(name, "-> R2:", round(r2_score(y_test, pred), 4),
          " RMSE:", round(np.sqrt(mean_squared_error(y_test, pred)), 4))